# Lab 1 · Part 1 — Calibration check

**Goal:** Load the saved JSON file and check the steering and throttle with a joystick.

This is a short validation notebook. It does **not** change or save the calibration.

**Before you start:** Stop the car, turn off the motor power, and shut down the calibration notebook kernel. Keep the area clear. Use the physical power switch if the car does not stop.

## 1. Load the JSON file

Put `jetracer_calibration.json` in the same folder as this notebook. Confirm that the car ID and saved values are correct.

In [1]:
import json
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display
from jetracer.nvidia_racecar import NvidiaRacecar

CONFIG_FILE = Path('jetracer_calibration.json')

with CONFIG_FILE.open('r') as f:
    settings = json.load(f)

if not settings['steering_calibrated']:
    raise ValueError('Steering is not yet calibrated. Finish the first notebook.')

print('Car ID:', settings['car_id'])
print('Steering gain:', settings['steering_gain'])
print('Steering offset:', settings['steering_offset'])
print('Throttle gain:', settings['throttle_gain'])


Car ID: jetracer_1
Steering gain: -0.9
Steering offset: 0.09
Throttle gain: 0.8


WARNNIG: Jetson.GPIO library has not been verified with this carrier board,


## 2. Apply the saved calibration

The JetRacer starts with throttle set to `0.0`. The notebook then applies the values read from JSON.

In [2]:
car = NvidiaRacecar()
car.throttle = 0.0
car.steering_gain = float(settings['steering_gain'])
car.steering_offset = float(settings['steering_offset'])
car.throttle_gain = float(settings['throttle_gain'])
car.steering = 0.0
car.steering_motor.throttle = car.steering_offset

print('Calibration loaded. Throttle is 0.')


Calibration loaded. Throttle is 0.


## 3. Connect the joystick

1. Connect a gamepad to the computer running the web browser.
2. Press a button on the gamepad.
3. Check that its axes appear in the controller widget.

This notebook uses **axis 0** for steering and **axis 3** for throttle, as tested on this JetRacer setup. Your gamepad may use different axes.

In [3]:
controller = widgets.Controller(index=0)
display(controller)


Controller()

## 4. Test the controls

**First test:** Lift the drive wheels above the table. With **Enable joystick throttle** unticked, move the steering stick left and right. The front wheels must respond correctly.

**Second test:** After the instructor approves the test area, place the car on the floor. Keep the joystick throttle at neutral. Tick **Enable joystick throttle**. Use a small stick movement and then return to neutral.

The throttle is limited to the saved **NORMAL** value (`0.13` in the sample file). Reverse is disabled in this test. The **STOP** button disables throttle, but the physical power switch remains the emergency stop.

In [4]:
assert len(controller.axes) > 3, 'Connect the joystick before you continue.'

JOYSTICK_STEERING_SIGN = -1.0  # Change to 1.0 if steering is reversed.
enable_drive = widgets.Checkbox(value=False, description='Enable joystick throttle')
stop_button = widgets.Button(description='STOP', button_style='danger')
message = widgets.HTML(value='Throttle is stopped.')

def update_car(change=None):
    # Steering: invert joystick axis to match this car's gamepad.
    steering = JOYSTICK_STEERING_SIGN * float(controller.axes[0].value)
    if abs(steering) < 0.05:
        steering = 0.0
    steering = max(float(settings['steering_min']),
                   min(float(settings['steering_max']), steering))
    car.steering = steering

    # Throttle: forward only. Limit to the saved normal speed command.
    if enable_drive.value:
        forward = max(0.0, -float(controller.axes[3].value))
        throttle = forward * float(settings['joystick_throttle_gain'])
        throttle = min(float(settings['normal']), throttle)
        car.throttle = throttle
    else:
        car.throttle = 0.0

def on_drive_changed(change):
    if change['new'] and abs(float(controller.axes[3].value)) > 0.10:
        enable_drive.value = False
        message.value = 'Return the throttle stick to neutral. Then enable it.'
        return
    update_car()
    message.value = 'Joystick throttle enabled.' if enable_drive.value else 'Throttle is stopped.'

def stop_car(button):
    enable_drive.value = False
    car.throttle = 0.0
    message.value = 'Throttle is stopped.'

controller.axes[0].observe(update_car, names='value')
controller.axes[3].observe(update_car, names='value')
enable_drive.observe(on_drive_changed, names='value')
stop_button.on_click(stop_car)

car.throttle = 0.0
display(widgets.VBox([enable_drive, stop_button, message]))


## 5. End the test

Check that:

- The wheels are straight at joystick center.
- Left and right directions are correct.
- The car starts slowly when you enable throttle.
- Returning the throttle stick to neutral stops the car.

Press **STOP**. Then run the next cell and turn off the car.

In [ ]:
enable_drive.value = False
car.throttle = 0.0
car.steering = 0.0

controller.axes[0].unobserve(update_car, names='value')
controller.axes[3].unobserve(update_car, names='value')

print('Test complete. Turn off the car power.')
